# Outline

- [ ] Greedy algorithm templates
- [ ] Typical scenarios for greedy algorithms
- [ ] Time and space complexity
- [ ] Comparison with other algorithms

# Patterns: When to Use Greedy

## How greedy works
Make the locally optimal choice at each step and never reconsider it -- no backtracking, no exploring alternatives. This only produces a globally optimal answer if the problem has:

1. **Greedy-choice property** -- a global optimum can be reached by making a series of locally optimal choices; the choice made now is never revisited or second-guessed later.
2. **Optimal substructure** -- an optimal solution to the whole problem contains optimal solutions to its subproblems (DP needs this too).

**Greedy vs DP:** both need optimal substructure, but DP explores *multiple* choices per subproblem and combines the best (hence the table/memo), while greedy commits to *one* choice and moves on. That's why greedy is cheaper (usually O(n log n) from sorting + one pass) when it's valid, and silently wrong when it isn't -- there's no fallback/reconsideration built in.

**Proving greedy is correct** (do this out loud before coding -- it's the actual interview signal, not just getting the right answer):
- **Exchange argument:** assume an optimal solution differs from the greedy one at the first choice; show that swapping in the greedy choice doesn't make the solution worse. Repeat inductively -> greedy solution is at least as good as any optimal one.
- **Staying-ahead argument:** show that after every step, the greedy partial solution is at least as good as any other valid partial solution, by some measurable metric (e.g. greedy's k-th interval always ends no later than any other valid choice's k-th interval).
- If you can't sketch this argument in about a minute, treat it as a signal greedy might be wrong -- fall back to DP rather than hoping.

## When to use it (signals)
- "minimum number of X to cover/reach Y" -- jump game / minimum jumps, interval covering, coin change *with a canonical coin system*.
- "maximum number of Y you can pick/afford under a budget/capacity constraint" -- sort by cost/size, take the cheapest/smallest first (ice cream bars, assign cookies, fractional knapsack).
- "non-overlapping intervals," "merge/schedule intervals," "minimum meeting rooms/platforms" -- sort by start or end time, sweep once.
- "maximize/minimize by choosing an order" -- sort by a derived key (ratio, difference, end time, deadline) then make one pass.
- Sorting is very often a prerequisite: greedy almost always *sorts by something first*, then makes a single linear pass with no backtracking.

## When NOT to use it (reach for DP instead)
- The problem lets earlier choices' value depend on later choices (interacting decisions) -- the exchange argument fails because swapping isn't locally safe.
- "Count the number of ways" -- greedy only ever produces *one* answer/path, it can't enumerate or count.
- You can find a counterexample by hand-tracing 3-4 elements -- if you can break your greedy rule that easily, it's wrong; don't force it, switch to DP.
- Classic gotcha -- **coin change with arbitrary denominations**: coins `[1, 3, 4]`, target `6` -> greedy (largest first) picks `4+1+1` = 3 coins, but the optimal is `3+3` = 2 coins. Greedy only works here for "canonical" coin systems (like US currency); in general, use DP.

## Typical template
```python
def greedy_solve(items):
    items.sort(key=lambda x: some_key(x))   # the crux: what to sort by, and why that key is safe
    result = []
    state = init_state()
    for item in items:
        if fits(state, item):                # locally optimal / feasible choice, made once
            take(state, item)
            result.append(item)
    return result
```

## Quick decision checklist
| Question | If yes -> |
|---|---|
| Can you sort by some key and make one linear pass? | Try greedy first |
| Can you sketch the exchange/staying-ahead argument in under a minute? | Greedy is likely correct |
| Do later choices change the value/validity of earlier ones? | Greedy likely wrong -> DP |
| Found a counterexample by hand-tracing a small case? | Definitely DP (or a different sort key) |
| Asked to "count the number of ways"? | Not greedy -> DP |

From this notebook's list: 1833 (ice-cream bars -- sort ascending, buy cheapest first), 942 (Di String Match -- greedy low/high pointer assignment), 1846 (max element after decreasing/rearranging -- sort, cap each at prev+1), 1817 (minimize max pair sum -- sort, pair smallest with largest).

# Leetcode Examples


- [(medium)11 Container With Most Water](https://leetcode-cn.com/problems/container-with-most-water/)
- [(easy)553 Optimal Division](https://leetcode-cn.com/problems/optimal-division/)
- [(easy)942 Di String Match](https://leetcode.cn/problems/di-string-match/)
- [(medium)1218 Longest Arithmetic Subsequence of Given Difference](https://leetcode-cn.com/problems/longest-arithmetic-subsequence-of-given-difference/)
- [(medium)1708 Maximum Number of Eaten Apples](https://leetcode-cn.com/problems/maximum-number-of-eaten-apples/)
- [(hard)1713 Minimum Operations to Make a Subsequence](https://leetcode-cn.com/problems/minimum-operations-to-make-a-subsequence/)
- [(easy)1736 Latest Time by Replacing Hidden Digits](https://leetcode-cn.com/problems/latest-time-by-replacing-hidden-digits/)
- [(medium)1833 Maximum Ice-cream Bars](https://leetcode-cn.com/problems/maximum-ice-cream-bars/)
- [(medium)1846 Maximum Element After Decreasing and Rearranging](https://leetcode-cn.com/problems/maximum-element-after-decreasing-and-rearranging/)
- [(medium)1817 Minimize Maximum Pair Sum in Array](https://leetcode-cn.com/problems/minimize-maximum-pair-sum-in-array/)



## Worked Example: [(medium) 1833 Maximum Ice-Cream Bars](https://leetcode-cn.com/problems/maximum-ice-cream-bars/)

**Problem:** given `costs[i]` (price of the i-th ice cream bar) and `coins` (your budget), buy as many bars as possible (each bar at most once, any order) without exceeding the budget. Return the maximum count.

**Why this is a clean greedy example:** to maximize the *count* of items bought under a fixed budget, buying the cheapest available item first is never worse than any other order -- if an optimal solution skips a cheaper bar in favor of pairing two others, you could always swap in the cheaper one and still afford at least as much, possibly freeing coins for another bar (that's the exchange argument, made concrete). Sort ascending, then greedily buy while affordable.

In [ ]:
# 1833 -- Brute force: try every subset, keep the largest affordable one
# Time: O(2^n * n) -- 2^n subsets, O(n) to sum each
# Space: O(n) for the recursion / subset storage
from itertools import combinations
from typing import List

class SolutionBruteForce:
    def maxIceCream(self, costs: List[int], coins: int) -> int:
        n = len(costs)
        best = 0
        # only feasible for small n -- real constraints go up to 10^5
        for size in range(n, 0, -1):
            for combo in combinations(costs, size):
                if sum(combo) <= coins:
                    best = max(best, size)
                    break  # no combo of this size can beat `size` bars
        return best

print(SolutionBruteForce().maxIceCream([1, 3, 2, 4, 1], 7))  # 4
print(SolutionBruteForce().maxIceCream([10, 6, 8, 7, 7, 8], 5))  # 0

In [ ]:
# 1833 -- Optimal: sort ascending, greedily buy the cheapest remaining bar
# Time: O(n log n) -- dominated by the sort; the scan afterward is O(n)
# Space: O(1) extra (O(log n)-O(n) for the sort itself, depending on implementation)
#
# Key insight (exchange argument): to maximize *count* under a budget, buying
# the cheapest bar first is always at least as good as any other order --
# swapping a more expensive early pick for a cheaper one never reduces how
# many more bars you can still afford.

class SolutionOptimal:
    def maxIceCream(self, costs: List[int], coins: int) -> int:
        costs.sort()
        count = 0
        for cost in costs:
            if cost > coins:
                break  # sorted ascending -> every later bar is also unaffordable
            coins -= cost
            count += 1
        return count

print(SolutionOptimal().maxIceCream([1, 3, 2, 4, 1], 7))  # 4
print(SolutionOptimal().maxIceCream([10, 6, 8, 7, 7, 8], 5))  # 0

# My Summary
